# SRE agent for incident response

An incident alert opens a Slack thread in `#oncall`. The agent investigates production telemetry, GitHub pull requests and commits, AWS infrastructure, and similar past incidents. It posts its findings and asks a responder to approve any rollback.

```mermaid
sequenceDiagram
    participant Alert as PagerDuty / incident.io
    participant Slack as Slack #oncall
    participant App as Incident bot
    participant Agent as Agents API
    participant Sandbox as Incident sandbox + AWS skills
    participant Evidence as GitHub + AWS + incident history
    Alert->>App: Send production incident
    App->>Slack: Open an incident thread
    App->>Agent: Create one persistent incident session
    App->>Sandbox: Start codex exec-server with preinstalled AWS skills
    Agent->>Sandbox: Read relevant skills and the mounted runbook
    Agent->>Evidence: Inspect code, infrastructure, and previous incidents
    Evidence-->>Agent: Return correlated operational evidence
    Agent-->>App: agent.session.action_required webhook
    App->>Agent: Retrieve the pending function call
    App->>Slack: Post rollback approval buttons
    Slack-->>App: Approve or reject
    App->>Agent: Submit the decision as a tool result
    Agent-->>Slack: Explain the decision and next steps
    Slack-->>App: Ask a follow-up
    App->>Agent: Reuse the incident session for follow-ups
    App->>Agent: Save the resolution and close the session
    App->>Sandbox: Remove the incident sandbox
```


## Agents API capabilities

Function tools, Skills, Sandbox, Webhooks, Persistent sessions, Streaming.

### Incident-triggered execution

A PagerDuty, incident.io, or Alertmanager webhook starts the investigation automatically, without waiting for a responder to restate the symptoms.

### Persistent incident memory

One agent session retains the current incident's findings and follow-ups; a separate incident-history tool brings relevant past outages and mitigations into the investigation.

### Operational tools

Your application supplies service evidence and incident records. Service-connected MCP servers provide GitHub and AWS access, and the sandbox holds the runbook.

### Human-approved changes

The agent posts its findings and proposed rollback to #oncall, but only a responder can approve a production change.


## Application flow

1. PagerDuty / incident.io.
2. Slack #oncall.
3. Agents API session.
4. Sandbox + AWS skills.
5. GitHub / AWS / memory.
6. Approved recovery.


## What you need

- Python 3.14+ and `uv`.
- A sandbox: self-hosted Docker or a [third-party provider](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers).
- An OpenAI API key and a separate restricted executor key.
- A Slack app installed in `#oncall`, with a bot token, signing secret, and message event subscriptions.
- PagerDuty, incident.io, or an Alertmanager-compatible monitoring system.
- Read access to the affected GitHub repository.
- AWS DevOps Agent credentials or another read-only AWS operational integration.

The first run uses bundled metrics, logs, deployments, GitHub changes, AWS telemetry, and incident history for `checkout-api`. Tool results label this sample data; it is not a live production diagnosis. Slack delivery is real. Replace the evidence tools when connecting your own incidents.


## Configure the Slack bot

From the repository root:

```bash
cp examples/agents_api/apps/sev_bot/.env.example examples/agents_api/apps/sev_bot/.env
docker build -t agent-api-sev-sandbox:latest examples/agents_api/apps/sev_bot
```

Create the Slack app from [`slack-app-manifest.yaml`](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/sev_bot/slack-app-manifest.yaml). Replace `https://your-app.example` with your application's HTTPS address, install the app, and invite it to `#oncall`.

Add `OPENAI_API_KEY`, `OPENAI_EXECUTOR_API_KEY`, `SLACK_BOT_TOKEN`, and `SLACK_SIGNING_SECRET` to `examples/agents_api/apps/sev_bot/.env`. Use OpenAI keys with the same owner, organization, and project.

The manifest subscribes to `message.channels` and `message.groups` at `/slack/events`, and sends approval actions to `/slack/actions`. Slack must be able to reach both URLs over HTTPS. Posting the initial investigation only needs the bot token; follow-ups and approval buttons also require these callbacks.

Each incident gets a self-hosted sandbox running `codex exec-server`. The app passes only `OPENAI_EXECUTOR_API_KEY` as `CODEX_API_KEY`; the application, Slack, GitHub, and AWS credentials stay outside the sandbox. The executor key needs `api.agents.environments.connect` and IP restrictions that allow the sandbox's outbound network. Follow-ups reuse the sandbox, and resolution or application shutdown deletes it.

To create an executor key with the required permission, open [Agents > Environments > Keys](https://platform.openai.com/agents?tab=environments&environment_view=keys) and select **Create**.

The app mounts [`runbooks/`](https://github.com/openai/openai-cookbook/tree/main/examples/agents_api/apps/sev_bot/runbooks) read-only at `/workspace/runbooks`. The agent reads `checkout-api.md` for mitigation steps and recovery checks. Add a runbook named after each service when connecting your own incidents.


## Connect Agents API approval webhooks

In [Project settings > Webhooks](https://platform.openai.com/settings/project/webhooks), register `https://your-app.example/webhooks/openai`, subscribe to `agent.session.action_required`, and add its signing secret as `OPENAI_WEBHOOK_SECRET` in `.env`.

With these credentials configured, start the receiver:

```bash
uv run examples/agents_api/apps/sev_bot/main.py
```

The app verifies OpenAI's signature, retrieves the session's `required_actions`, and posts Slack buttons for `propose_rollback`. It saves the pending `turn_id` and `call_id` and ignores repeated deliveries for the same proposal. Read-only tools still run through the streaming handler; `propose_rollback` deliberately has no automatic handler.

The Slack callback returns the decision to the waiting call:

The corresponding implementation is included in the Python code cells below.

The agent continues after approval or rejection. Approval is not execution: the example never contacts a deployment system. Keep the app running to receive callbacks and stream the final update. OpenAI and Slack callbacks both require a reachable HTTPS address. Persist pending actions and use a durable worker when deploying beyond this single-process example.


## Send a sample incident

From another terminal, send the included alert to the receiver:

```bash
curl -X POST http://127.0.0.1:8003/webhooks/alerts \
  -H 'Content-Type: application/json' \
  --data-binary @examples/agents_api/apps/sev_bot/sample_alert.json
```

The agent posts its sample investigation to `#oncall`, tracing the outage to the fixture's pull request #418 and Redis pool exhaustion. Approve or reject the proposed rollback using the buttons in the Slack thread. Approval records a decision; it does not execute a deployment.

To inspect a real repository, configure GitHub MCP below.


## Connect incident alerts

Send incident webhooks through your authenticated ingress to:

```text
https://your-app.example/webhooks/alerts
```

For Alertmanager:

```yaml
receivers:
  - name: incident-bot
    webhook_configs:
      - url: https://your-app.example/webhooks/alerts
        http_config:
          authorization:
            credentials: your-shared-token
```

Set `ALERT_WEBHOOK_TOKEN` to the same value. Alertmanager sends it as a bearer token. Alert fingerprints deduplicate active incidents; a later alert can start a new investigation after the earlier incident is resolved.

For PagerDuty, subscribe to `incident.triggered` and `incident.resolved`, and match the PagerDuty service name to an entry in `operations.json`. For incident.io, subscribe to `public_incident.incident_created_v2` and `public_incident.incident_status_updated_v2`; set `INCIDENT_SERVICE` to the affected service for that subscription. Adapt this mapping for multi-service incidents.

The example parses these provider payloads but does not verify their native signatures. Your ingress must verify PagerDuty or incident.io signatures before forwarding them with `ALERT_WEBHOOK_TOKEN`. Slack signatures are verified by the application.


## Use AWS skills

The [Dockerfile](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/sev_bot/Dockerfile) installs the Codex CLI and all [AWS Agent Toolkit skills](https://github.com/aws/agent-toolkit-for-aws/tree/main/skills) during the image build. From `/workspace`, it runs:

```bash
npx --yes skills add aws/agent-toolkit-for-aws/skills \
  --agent codex --copy --yes
```

The skills are copied to `/workspace/.agents/skills` for automatic discovery. The agent selects relevant skills and loads their instructions as needed. Ask in the incident's Slack thread: "List the installed AWS skills and explain which ones help investigate Redis connection exhaustion."

Skills provide instructions, not AWS access. No cloud credentials are needed to read them. The sample evidence tools stay in the application; use read-only integrations for live AWS evidence. Rebuild the image to refresh its skills, and review third-party instructions before granting production access.


## Connect GitHub

Set `GITHUB_TOKEN` and `GITHUB_REPOSITORY` in `.env`. Use a fine-grained token limited to the affected repository, with read access to **Contents** and **Pull requests**.

The app connects to [GitHub MCP](https://github.com/github/github-mcp-server) through its read-only endpoint, `https://api.githubcopilot.com/mcp/readonly`. Its allowlist contains five tools: `list_commits`, `get_commit`, `list_pull_requests`, `pull_request_read`, and `get_file_contents`. Agents API calls these directly; no application handler or GitHub CLI installation is needed.

GitHub credentials stay outside the sandbox. If the configured MCP server cannot connect, the session fails rather than silently skipping repository inspection. Without `GITHUB_TOKEN`, `get_service_evidence` returns the bundled sample PRs and commits instead.


## Connect AWS

The [AWS Agent Toolkit](https://github.com/aws/agent-toolkit-for-aws) includes observability skills and an AWS DevOps Agent MCP server for infrastructure investigations. Uncomment `DEVOPS_AGENT_REGION` and `DEVOPS_AGENT_TOKEN` in `examples/agents_api/apps/sev_bot/.env`:

```text
DEVOPS_AGENT_REGION=us-east-1
DEVOPS_AGENT_TOKEN=...
```

The application registers the service-connected MCP server automatically:

The corresponding implementation is included in the Python code cells below.

Without AWS MCP configured, `get_service_evidence` includes bundled CloudWatch, ECS, and ElastiCache telemetry. With `DEVOPS_AGENT_TOKEN` set, it omits that sample AWS data so the agent uses the MCP server for AWS evidence. The service metrics, logs, and deployments remain sample data until you connect your monitoring system.


## Incident memory

Memory works at two levels:

- **Within an incident:** The alert fingerprint and Slack thread map to one Agents API session and sandbox. Follow-up questions retain previous evidence, tool results, investigation context, and workspace files.
- **Across incidents:** `recall_incidents` searches prior reports for related symptoms, root causes, and resolutions. A resolved alert saves findings to `examples/agents_api/apps/sev_bot/incident_memory.json` before the session closes. The app reloads this file on startup, or starts with `incident_history.json` when no saved file exists.

The generated memory file is ignored by Git and replaced atomically on each save. Deleting it resets history to the bundled samples. Run one app process against this file; it is not a shared database.

Only resolved-incident history survives restarts. Active Slack threads, session IDs, pending approvals, and sandbox handles still live in memory. Approving a rollback records the decision; this example never deploys or changes production infrastructure.


## Investigation tools

- `get_service_evidence` returns metrics, recent error logs, and deployments together.
- `recall_incidents` retrieves relevant incident history and prior mitigations.
- `propose_rollback` requests human approval without changing production.

Only the first two have automatic handlers. The webhook and Slack callback complete `propose_rollback` after a responder decides. GitHub and AWS use MCP integrations; runbooks are files in the sandbox.


## Run this notebook

Use a Jupyter Python kernel (Python 3.11 or later) on macOS or Linux in a local clone of the [Cookbook repository](https://github.com/openai/openai-cookbook). The application itself uses Python 3.14; `uv run` installs the dependencies declared in `main.py` and selects that interpreter.

The terminal commands above run the checked-in application. The notebook instead builds a separate copy inside an ignored `tmp_` workspace under your Cookbook checkout. Each `%%writefile` cell contains actual application source. Run these cells in order: the first cell for a module creates its file, and later cells append to it. Python definitions are executed by the application when you launch it.

The setup cell copies only the listed supporting fixtures, manifests, and policy files. It creates a fresh `.env` from the example template without copying your existing credentials. Configure the printed `.env` path before the optional launch step. Rerunning setup creates a new workspace; keep the previous workspace if you need its reports or memory.

Default execution builds and checks the files locally. Docker builds and live API calls require the explicit flags in the launch section.


In [ ]:
from pathlib import Path
import shutil
import tempfile

if globals().get("application_process") is not None and application_process.poll() is None:
    raise RuntimeError("Stop the running application before creating a new workspace.")
application_process = None

# Start Jupyter anywhere inside the Cookbook checkout.
working_directory = Path.cwd().resolve()
cookbook_root = next(
    (path for path in [working_directory, *working_directory.parents]
     if (path / "examples/agents_api/apps/sev_bot/main.py").is_file()),
    None,
)
if cookbook_root is None:
    raise FileNotFoundError("Clone openai/openai-cookbook and start Jupyter inside it.")

notebook_root = Path(tempfile.mkdtemp(prefix="tmp_agents_sev_bot_", dir=cookbook_root))
application_dir = notebook_root / "examples/agents_api/apps/sev_bot"
application_dir.mkdir(parents=True)
for package in [notebook_root / "examples", notebook_root / "examples/agents_api",
                notebook_root / "examples/agents_api/apps", application_dir]:
    (package / "__init__.py").touch()

support_paths = [
    ".dockerignore",
    ".env.example",
    "Dockerfile",
    "incident_history.json",
    "operations.json",
    "runbooks/checkout-api.md",
    "sample_alert.json",
    "slack-app-manifest.yaml"
]
source_dir = cookbook_root / "examples/agents_api/apps/sev_bot"
for relative_path in support_paths:
    destination = application_dir / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_dir / relative_path, destination)
shutil.copy2(application_dir / ".env.example", application_dir / ".env")
shutil.copytree(
    cookbook_root / "examples/agents_api/sandboxes/application_managed/docker",
    notebook_root / "examples/agents_api/sandboxes/application_managed/docker",
)
print(f"Application workspace: {application_dir}")
print(f"Configure credentials in: {application_dir / '.env'}")


## Implementation walkthrough

This walkthrough covers incident webhooks, Slack updates, GitHub and AWS investigation tools, persistent incident memory, and human-approved recovery.

Follow the setup instructions above, then build the application with the code cells below.


### 1. Set up

Clone the repository, configure your OpenAI API key, separate restricted executor key, and Slack app credentials, and build the incident sandbox image.

Create the Slack app from examples/agents_api/apps/sev_bot/slack-app-manifest.yaml. Set its HTTPS URLs to your host, install it, and invite it to `#oncall`. The manifest routes message.channels and message.groups to /slack/events and approval buttons to /slack/actions.


### 2. Install AWS skills in the sandbox

The sandbox image contains Python, the Codex CLI, and AWS Agent Toolkit skills. Install the skills once at build time so every incident starts with the same guidance.

This copies all skills into /workspace/.agents/skills for automatic discovery.

Read the implementation in [Dockerfile](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/sev_bot/Dockerfile).


### 3. Receive incident alerts

Accept webhooks from PagerDuty, incident.io, or Alertmanager and normalize them into a common incident shape. Start the investigation in the background and reuse the alert fingerprint to avoid duplicate incidents.

The complete example's `queue_alerts` helper normalizes payloads, deduplicates active alerts, retries failed investigations, and handles resolved incidents.


### 4. Inspect the alert

The normalized alert identifies the affected service, severity, and customer-visible symptom. Its fingerprint becomes the stable key for both the agent session and the Slack incident thread.


### 5. Open the Slack incident thread

Post the incoming incident to `#oncall` and keep the returned thread timestamp. Investigation findings, follow-up questions, and rollback approvals all stay in this thread.


### 6. Define the application tools

Combine metrics, logs, and deployments in `get_service_evidence`. Use `recall_incidents` for past outages and `propose_rollback` to request approval. Only the two read-only functions run automatically; GitHub and AWS use MCP.

Keep `propose_rollback` out of handlers so its call stays pending until Slack approval. The first run uses bundled sample telemetry; connect `get_service_evidence` to your monitoring system before diagnosing real incidents.


### 7. Connect GitHub MCP

Set `GITHUB_TOKEN` and `GITHUB_REPOSITORY` in .env. Use a fine-grained token scoped to the affected repository, with read access to Contents and Pull requests. Agents API calls GitHub directly, without a custom handler or sandbox credentials.

The read-only endpoint and allowlist expose only repository inspection tools. `GITHUB_REPOSITORY` tells the agent where to look; token permissions enforce access. Without `GITHUB_TOKEN`, the runnable example skips MCP and includes bundled sample PRs and commits in `get_service_evidence`.


### 8. Connect the AWS DevOps Agent

The AWS Agent Toolkit includes an incident-response MCP server. Configure it as a service-connected tool so the agent can investigate AWS infrastructure without moving cloud credentials into a sandbox.

Without AWS MCP configured, `get_service_evidence` includes bundled CloudWatch, ECS, and ElastiCache telemetry. Setting `DEVOPS_AGENT_TOKEN` omits that sample AWS data and enables the MCP server.


### 9. Create an incident session

Create one self-hosted Agents API session per alert fingerprint. Have the agent correlate evidence and request approval before recovery.


### 10. Connect the incident sandbox

Mount examples/agents_api/apps/sev_bot/runbooks read-only at /workspace/runbooks. Start codex exec-server with the session's environment ID, then submit the investigation. Keep the container alive for follow-ups.

Inject `OPENAI_EXECUTOR_API_KEY` as `CODEX_API_KEY` at runtime, never during the image build. The example keeps application, Slack, GitHub, and AWS credentials outside the sandbox.


### 11. Receive approval requests from Agents API

Register /webhooks/openai in your OpenAI project's webhook settings, subscribe to agent.session.action_required, and set `OPENAI_WEBHOOK_SECRET`. Retrieve the pending call before asking for a Slack decision.

The runnable receiver verifies signatures, validates the proposed service and version, and handles repeated deliveries. Do not register an automatic handler for `propose_rollback`: the function call must remain pending until a person decides.


### 12. Post findings to `#oncall`

Save the session ID, track tool activity, and publish the completed investigation into the original Slack thread. The diagnosis can cite a pull request, commit, CloudWatch alarm, and related prior incident.


### 13. Use incident memory

The incident session retains its conversation. Across incidents, `recall_incidents` searches history loaded from a local JSON file. On the first run, load the bundled sample history instead.

Resolved-incident history survives restarts in `incident_memory.json`, which is ignored by Git. Active Slack threads, approvals, session IDs, and sandbox handles still live in memory. Use one app process per file.


### 14. Return the human decision to the agent

The Slack callback submits approval or rejection as the pending function's result. The agent continues the same turn and explains the next steps. No deployment is executed.

The runnable callback acknowledges Slack immediately and submits the result in the background. Both OpenAI and Slack callbacks need reachable HTTPS URLs. The original stream stays open for progress and resumed output; approval is driven by the webhook.


### 15. Start the Slack incident bot

Start the webhook receiver, then send the included alert from another terminal. Investigation updates, follow-ups, and approval buttons appear in `#oncall`.


### 16. Connect your incident provider

Forward authenticated incident events to the receiver. For PagerDuty, subscribe to incident.triggered and incident.resolved and match its service name to operations.json. For incident.io, subscribe to public incident-created and status-updated v2 events and set `INCIDENT_SERVICE` for the subscription.

Set `ALERT_WEBHOOK_TOKEN` to the same value. For PagerDuty or incident.io, your ingress must verify the provider's native signature and forward a bearer token; the example does not implement that signature verification.


### 17. Resolve the incident

A resolved alert saves the findings, posts a final Slack update, and deletes the session and sandbox. Write a temporary file, then replace the saved history atomically. Approval alone does not mean a rollback ran.

The runnable example also cleans up the sandbox after failed investigations and on application shutdown.


## Build the application

The following cells include every application module. Run all cells for each file before launching. The inline dependency declaration in `main.py` installs the current OpenAI SDK and the application libraries through `uv`.


### alerts.py

Normalize incoming incidents and route supported alert formats into the investigation queue.


In [ ]:
%%writefile "{application_dir}/alerts.py"
"""Normalize incident-provider alerts and schedule independent investigations."""

from __future__ import annotations

import asyncio
import logging
import os
from typing import Any

from fastapi import BackgroundTasks, HTTPException

from .agent import IncidentBot

logger = logging.getLogger(__name__)


def normalize_alerts(payload: dict[str, Any]) -> list[dict[str, Any]]:
    alerts = payload.get("alerts")
    if isinstance(alerts, list):
        if not all(isinstance(alert, dict) for alert in alerts):
            raise HTTPException(status_code=400, detail="Each alert must be an object.")
        return [
            {
                **alert,
                "status": alert.get("status", payload.get("status", "firing")),
                "fingerprint": alert.get("fingerprint")
                or f"{alert.get('labels', {}).get('service', '')}:"
                f"{alert.get('labels', {}).get('alertname', '')}",
            }
            for alert in alerts
        ]

    event = payload.get("event")
    if isinstance(event, dict) and isinstance(event.get("data"), dict):
        event_type = str(event.get("event_type", ""))
        if event_type not in {"incident.triggered", "incident.resolved"}:
            return []
        data = event["data"]
        service = data.get("service", {})
        details = data.get("custom_details", {})
        service_name = (
            details.get("service") or service.get("name") or service.get("summary")
        )
        return [
            {
                "fingerprint": f"pagerduty:{data['id']}",
                "status": "resolved" if event_type.endswith(".resolved") else "firing",
                "labels": {
                    "service": service_name,
                    "severity": "critical"
                    if data.get("urgency") == "high"
                    else "warning",
                    "alertname": event_type,
                },
                "annotations": {
                    "summary": str(data.get("title") or data.get("summary", ""))
                },
            }
        ]

    incident_event_type = payload.get("event_type")
    if isinstance(incident_event_type, str):
        if incident_event_type not in {
            "public_incident.incident_created_v2",
            "public_incident.incident_status_updated_v2",
        }:
            return []
        data = payload[incident_event_type]
        data = data.get("incident", data)
        service_name = os.environ.get("INCIDENT_SERVICE")
        if not service_name:
            raise HTTPException(
                status_code=400,
                detail="Set INCIDENT_SERVICE for this incident.io subscription.",
            )
        category = data.get("incident_status", {}).get("category", "live")
        return [
            {
                "fingerprint": f"incidentio:{data['id']}",
                "status": (
                    "resolved"
                    if category
                    in {"learning", "closed", "declined", "merged", "canceled"}
                    else "firing"
                ),
                "labels": {
                    "service": service_name,
                    "severity": (
                        "critical"
                        if (data.get("severity") or {}).get("name", "").lower()
                        in {"critical", "sev-1"}
                        else "warning"
                    ),
                    "alertname": incident_event_type,
                },
                "annotations": {"summary": str(data.get("name", ""))},
            }
        ]

    raise HTTPException(
        status_code=400,
        detail="Expected an Alertmanager, PagerDuty, or incident.io incident webhook.",
    )




Continue `alerts.py`: `run_incident_tasks`, `queue_alerts`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/alerts.py"
async def run_incident_tasks(tasks: BackgroundTasks) -> None:
    # One incident waiting for approval must not hold up the rest of an alert batch.
    results = await asyncio.gather(
        *(task() for task in tasks.tasks), return_exceptions=True
    )
    for result in results:
        if isinstance(result, Exception):
            logger.error("Incident background task failed", exc_info=result)


def queue_alerts(
    bot: IncidentBot, payload: dict[str, Any], tasks: BackgroundTasks
) -> dict[str, Any]:
    alerts = normalize_alerts(payload)
    incident_tasks = BackgroundTasks()

    accepted: list[str] = []
    duplicates: list[str] = []
    for alert in alerts:
        fingerprint = str(alert.get("fingerprint", ""))
        existing = bot.fingerprints.get(fingerprint)
        if alert.get("status") == "resolved":
            if existing is not None:
                incident_tasks.add_task(bot.resolve, existing)
                accepted.append(existing)
            continue

        if existing is not None:
            incident = bot.incident(existing)
            if incident.status == "failed":
                incident.status = "investigating"
                incident_tasks.add_task(bot.investigate, incident)
                accepted.append(existing)
            else:
                duplicates.append(existing)
            continue

        try:
            incident = bot.start_incident(alert)
        except ValueError as error:
            raise HTTPException(status_code=400, detail=str(error)) from error
        accepted.append(incident.id)
        incident_tasks.add_task(bot.investigate, incident)

    if incident_tasks.tasks:
        tasks.add_task(run_incident_tasks, incident_tasks)

    return {
        "status": "accepted"
        if accepted
        else "already_tracking"
        if duplicates
        else "ignored",
        "incidents": accepted or duplicates,
    }


### memory.py

Load incident history and save recovery outcomes for later investigations.


In [ ]:
%%writefile "{application_dir}/memory.py"
"""Load, search, and atomically save resolved incident history."""

from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, cast


def load_history(path: Path) -> list[dict[str, Any]]:
    source = (
        path if path.exists() else Path(__file__).with_name("incident_history.json")
    )
    return cast(list[dict[str, Any]], json.loads(source.read_text()))


def save_history(path: Path, history: list[dict[str, Any]]) -> None:
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(history, indent=2) + "\n")
    temporary.replace(path)


def recall_incidents(
    history: list[dict[str, Any]], arguments: dict[str, Any]
) -> dict[str, Any]:
    name = str(arguments["service"])
    words = set(re.findall(r"[a-z0-9]+", str(arguments["query"]).lower()))
    candidates = [record for record in history if record.get("service") == name]
    ranked = sorted(
        candidates,
        key=lambda record: len(
            words.intersection(re.findall(r"[a-z0-9]+", json.dumps(record).lower()))
        ),
        reverse=True,
    )
    return {"service": name, "incidents": ranked[:3]}


### slack.py

Post incident updates and approval buttons, and verify signed Slack callbacks.


In [ ]:
%%writefile "{application_dir}/slack.py"
"""Publish incident messages and verify Slack callbacks."""

from __future__ import annotations

import hashlib
import hmac
import os
import time
from typing import TYPE_CHECKING, Any

from fastapi import HTTPException, Request
from slack_sdk.web.async_client import AsyncWebClient

if TYPE_CHECKING:
    from .agent import Incident

SLACK_CHANNEL = "#oncall"


class SlackChannel:
    """Post incident updates and approval buttons to a Slack thread."""

    def __init__(self, token: str | None = None, channel: str = SLACK_CHANNEL) -> None:
        self.channel = channel
        self.client = AsyncWebClient(token=token or os.environ["SLACK_BOT_TOKEN"])

    async def post(
        self,
        text: str,
        *,
        thread_ts: str | None = None,
        blocks: list[dict[str, Any]] | None = None,
    ) -> dict[str, Any]:
        payload: dict[str, Any] = {"channel": self.channel, "text": text}
        if thread_ts is not None:
            payload["thread_ts"] = thread_ts
        if blocks is not None:
            payload["blocks"] = blocks
        response = await self.client.chat_postMessage(**payload)
        return {"channel": str(response["channel"]), "ts": str(response["ts"])}


def approval_blocks(incident: Incident) -> list[dict[str, Any]]:
    if incident.action is None:
        raise ValueError("No rollback has been proposed.")
    return [
        {
            "type": "section",
            "text": {
                "type": "mrkdwn",
                "text": (
                    "*Rollback approval required*\n"
                    f"`{incident.action['service']}` to `{incident.action['version']}`\n"
                    f"{incident.action['reason']}\n"
                    "Approval records a decision; it does not execute a deployment."
                ),
            },
        },
        {
            "type": "actions",
            "elements": [
                {
                    "type": "button",
                    "text": {"type": "plain_text", "text": "Approve proposal"},
                    "style": "primary",
                    "action_id": "approve_rollback",
                    "value": incident.id,
                },
                {
                    "type": "button",
                    "text": {"type": "plain_text", "text": "Reject"},
                    "action_id": "reject_rollback",
                    "value": incident.id,
                },
            ],
        },
    ]


def verify_slack_request(body: bytes, request: Request) -> None:
    secret = os.environ.get("SLACK_SIGNING_SECRET")
    if not secret:
        raise HTTPException(status_code=500, detail="Set SLACK_SIGNING_SECRET.")

    timestamp = request.headers.get("x-slack-request-timestamp", "")
    signature = request.headers.get("x-slack-signature", "")
    try:
        if abs(time.time() - int(timestamp)) > 300:
            raise ValueError("Expired request.")
    except ValueError as error:
        raise HTTPException(
            status_code=401, detail="Invalid Slack request timestamp."
        ) from error

    digest = hmac.new(
        secret.encode(), b"v0:" + timestamp.encode() + b":" + body, hashlib.sha256
    ).hexdigest()
    if not hmac.compare_digest(signature, f"v0={digest}"):
        raise HTTPException(status_code=401, detail="Invalid Slack request signature.")


### tools.py

Define read-only evidence tools and optional GitHub and AWS MCP connections. The bundled operations data is a fixture.


In [ ]:
%%writefile "{application_dir}/tools.py"
"""Configure incident tools and read service evidence."""

from __future__ import annotations

import os
from typing import Any, cast

from openai.types.beta import AgentToolParam
from openai.types.beta.agent_tool_param import (
    AgentToolConfigParamFunction,
    AgentToolConfigParamMcp,
)


def define_tool(
    name: str, description: str, *fields: str
) -> AgentToolConfigParamFunction:
    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": {
            "type": "object",
            "properties": {field: {"type": "string"} for field in fields},
            "required": list(fields),
            "additionalProperties": False,
        },
    }


TOOLS: list[AgentToolParam] = [
    define_tool(
        "get_service_evidence",
        "Inspect service metrics, recent error logs, deployments, and bundled AWS evidence.",
        "service",
    ),
    define_tool(
        "recall_incidents",
        "Find related past incidents, root causes, and mitigations.",
        "service",
        "query",
    ),
    define_tool(
        "propose_rollback",
        "Request human approval to roll back to a known healthy deployment. Never executes it.",
        "service",
        "version",
        "reason",
    ),
    {"type": "programmatic_tool_calling", "enabled": True},
]
PROGRESS = {
    "get_service_evidence": "Checking service metrics, logs, and recent deployments.",
    "recall_incidents": "Retrieving related incident history.",
    "propose_rollback": "Preparing a rollback proposal for review.",
}


def configured_tools() -> list[AgentToolParam]:
    tools = [*TOOLS]
    github_token = os.environ.get("GITHUB_TOKEN")
    if github_token:
        github_tool: AgentToolConfigParamMcp = {
            "type": "mcp",
            "server_label": "github",
            "connection_origin": "service",
            "required": True,
            "allowed_tools": [
                "list_commits",
                "get_commit",
                "list_pull_requests",
                "pull_request_read",
                "get_file_contents",
            ],
            "transport": {
                "type": "http",
                "server_url": "https://api.githubcopilot.com/mcp/readonly",
                "authorization": f"Bearer {github_token}",
            },
        }
        tools.append(github_tool)
    token = os.environ.get("DEVOPS_AGENT_TOKEN")
    if token:
        region = os.environ.get("DEVOPS_AGENT_REGION", "us-east-1")
        aws_tool: AgentToolConfigParamMcp = {
            "type": "mcp",
            "server_label": "aws_devops",
            "connection_origin": "service",
            "transport": {
                "type": "http",
                "server_url": f"https://connect.aidevops.{region}.api.aws/mcp",
                "authorization": f"Bearer {token}",
            },
        }
        tools.append(aws_tool)
    return tools


def get_service(operations: dict[str, Any], name: str) -> dict[str, Any]:
    record = operations.get(name)
    if not isinstance(record, dict):
        raise ValueError(f"Unknown service: {name}")
    return cast(dict[str, Any], record)




Continue `tools.py`: `get_service_evidence`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/tools.py"
def get_service_evidence(
    operations: dict[str, Any], arguments: dict[str, Any]
) -> dict[str, Any]:
    name = str(arguments["service"])
    service = get_service(operations, name)
    evidence: dict[str, Any] = {
        "source": "bundled_sample",
        "service": name,
        "team": str(service["team"]),
        "metrics": service["metrics"],
        "logs": service["logs"][-20:],
        "deployments": service["deployments"],
    }
    if not os.environ.get("DEVOPS_AGENT_TOKEN"):
        evidence["aws"] = service["aws"]
    evidence["repository"] = str(service["repository"])
    if os.environ.get("GITHUB_TOKEN"):
        evidence["repository"] = os.environ.get(
            "GITHUB_REPOSITORY", str(service["repository"])
        )
    else:
        evidence["pull_requests"] = service["pull_requests"]
        evidence["commits"] = service["commits"]
    return evidence


### agent.py

Create persistent incident sessions, start their executors, stream investigations, and return human decisions to pending tool calls.


In [ ]:
%%writefile "{application_dir}/agent.py"
"""Investigate incidents and coordinate approval, resolution, and cleanup."""

from __future__ import annotations

import asyncio
import json
import logging
import os
import sys
from collections.abc import AsyncIterator
from dataclasses import dataclass, field
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import docker
from docker.models.containers import Container
from openai import AsyncOpenAI, NotFoundError
from openai.lib.streaming.agents import AsyncToolHandler
from openai.types.beta import AgentSessionEvent
from openai.types.beta.agents.session_create_params import Agent

from .memory import load_history, recall_incidents, save_history
from .slack import SLACK_CHANNEL, SlackChannel, approval_blocks
from .tools import PROGRESS, configured_tools, get_service, get_service_evidence

EXAMPLE_DIR = Path(__file__).resolve().parent
DOCKER_IMAGE = "agent-api-sev-sandbox:latest"
logger = logging.getLogger(__name__)
INSTRUCTIONS = """\
Investigate the incident using service evidence, GitHub, AWS, and past incidents.
Consult /workspace/runbooks/<service>.md and explain impact, likely cause, and next steps.
Use read-only tools; never install integrations or change infrastructure.
Clearly separate sample data from live findings.
If a rollout caused the incident, call propose_rollback once for the previous healthy version.
Approval records a decision, not an executed rollback.
"""


def start_executor(environment_id: str, remote_url: str) -> Container:
    return docker.from_env().containers.run(
        DOCKER_IMAGE,
        [
            "codex",
            "exec-server",
            "--remote",
            remote_url,
            "--environment-id",
            environment_id,
        ],
        environment={"CODEX_API_KEY": os.environ["OPENAI_EXECUTOR_API_KEY"]},
        volumes={
            str(EXAMPLE_DIR / "runbooks"): {"bind": "/workspace/runbooks", "mode": "ro"}
        },
        working_dir="/workspace",
        detach=True,
    )


@dataclass
class Incident:
    id: str
    fingerprint: str
    service: str
    severity: str
    title: str
    team: str
    channel: str = SLACK_CHANNEL
    status: str = "investigating"
    session_id: str | None = None
    sandbox: Container | None = field(default=None, repr=False)
    slack_thread_ts: str | None = None
    slack_approval_ts: str | None = None
    analysis: str = ""
    action: dict[str, Any] | None = None
    timeline: list[dict[str, Any]] = field(default_factory=list)
    lock: asyncio.Lock = field(default_factory=asyncio.Lock, repr=False)
    approval_lock: asyncio.Lock = field(default_factory=asyncio.Lock, repr=False)

    def record(self, message: str) -> None:
        self.timeline.append(
            {"time": datetime.now(UTC).strftime("%H:%M:%S UTC"), "message": message}
        )


class IncidentBot:
    """Keep one Agents API session and approval boundary for each incident."""

    def __init__(
        self,
        client: AsyncOpenAI,
        operations: dict[str, Any] | None = None,
        *,
        slack: SlackChannel | None = None,
        history: list[dict[str, Any]] | None = None,
        history_path: Path = EXAMPLE_DIR / "incident_memory.json",
    ) -> None:
        self.client = client
        self.model = os.environ.get("OPENAI_MODEL", "gpt-5.6-sol")
        self.operations = (
            operations
            if operations is not None
            else json.loads((EXAMPLE_DIR / "operations.json").read_text())
        )
        self.slack = slack if slack is not None else SlackChannel()
        self.history_path = history_path
        if history is not None:
            self.history = [*history]
        else:
            self.history = load_history(history_path)
        self.next_incident_number = (
            max(
                [1041]
                + [
                    int(str(record["id"]).removeprefix("INC-"))
                    for record in self.history
                ]
            )
            + 1
        )
        self.incidents: dict[str, Incident] = {}
        self.fingerprints: dict[str, str] = {}
        self.slack_threads: dict[str, str] = {}
        self.slack_deliveries: set[str] = set()
        self.closed_sessions: set[str] = set()



Continue `agent.py`: `IncidentBot.service`, `IncidentBot.start_incident`, `IncidentBot.get_service_evidence`, `IncidentBot.recall_incidents`, `IncidentBot.propose_rollback`, `IncidentBot.publish`, `IncidentBot.investigate`, `IncidentBot._investigate`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/agent.py"
    def service(self, name: str) -> dict[str, Any]:
        return get_service(self.operations, name)

    def start_incident(self, alert: dict[str, Any]) -> Incident:
        labels = alert.get("labels", {})
        annotations = alert.get("annotations", {})
        service = str(labels.get("service", ""))
        fingerprint = str(
            alert.get("fingerprint") or f"{service}:{labels.get('alertname', '')}"
        )
        existing = self.fingerprints.get(fingerprint)
        if existing is not None:
            return self.incidents[existing]

        service_info = self.service(service)
        severity = "SEV-1" if labels.get("severity") == "critical" else "SEV-2"
        incident = Incident(
            id=f"INC-{self.next_incident_number}",
            fingerprint=fingerprint,
            service=service,
            severity=severity,
            title=str(
                annotations.get("summary", labels.get("alertname", "Service alert"))
            ),
            team=str(service_info["team"]),
        )
        self.next_incident_number += 1
        incident.record(f"Received {severity} alert for {service}.")
        self.incidents[incident.id] = incident
        self.fingerprints[fingerprint] = incident.id
        return incident

    def get_service_evidence(self, arguments: dict[str, Any]) -> dict[str, Any]:
        return get_service_evidence(self.operations, arguments)

    def recall_incidents(self, arguments: dict[str, Any]) -> dict[str, Any]:
        return recall_incidents(self.history, arguments)

    def propose_rollback(
        self, incident: Incident, arguments: dict[str, Any]
    ) -> dict[str, Any]:
        service_name = str(arguments["service"])
        version = str(arguments["version"])
        if service_name != incident.service:
            raise ValueError("A rollback can only target the affected service.")

        deployments: list[dict[str, Any]] = self.service(service_name)["deployments"]
        healthy_versions = {
            str(deployment["version"])
            for deployment in deployments
            if deployment.get("status") == "previous_healthy"
        }
        if version not in healthy_versions:
            raise ValueError("A rollback must target a known healthy deployment.")

        if incident.action is None:
            incident.action = {
                "id": f"ACTION-{incident.id.removeprefix('INC-')}",
                "service": service_name,
                "version": version,
                "reason": str(arguments["reason"]),
                "status": "awaiting_approval",
            }
            incident.status = "awaiting_approval"
            incident.record(
                f"Requested approval to roll back {service_name} to {version}."
            )

        return incident.action

    async def publish(
        self,
        incident: Incident,
        text: str,
        *,
        blocks: list[dict[str, Any]] | None = None,
    ) -> dict[str, Any]:
        message = await self.slack.post(
            text, thread_ts=incident.slack_thread_ts, blocks=blocks
        )
        if incident.slack_thread_ts is None:
            incident.slack_thread_ts = str(message["ts"])
            self.slack_threads[incident.slack_thread_ts] = incident.id
        return message

    async def investigate(self, incident: Incident, question: str | None = None) -> str:
        async with incident.lock:
            if incident.status in {"resolved", "resolving"}:
                return "This incident is closing or already resolved."
            try:
                return await self._investigate(incident, question)
            except Exception as error:
                incident.status = "failed"
                incident.record(f"Investigation failed: {error}")
                await self.close_runtime(incident)
                raise

    async def _investigate(self, incident: Incident, question: str | None) -> str:
        handlers: dict[str, AsyncToolHandler] = {
            "get_service_evidence": self.get_service_evidence,
            "recall_incidents": self.recall_incidents,
        }
        prompt = (
            question
            or f"""\
Investigate {incident.severity}: {incident.title}
Affected service: {incident.service}

Inspect service health, production logs, recent deployments, GitHub pull requests and commits,
AWS infrastructure, related past incidents, and /workspace/runbooks/{incident.service}.md.
Correlate timestamps, quantify customer impact, and cite the responsible code.
If a rollout caused the incident, propose a rollback to the previous healthy version.
"""
        )

        if incident.session_id is not None:
            session = await self.client.beta.agents.sessions.retrieve(
                incident.session_id
            )
            incident.record("Received a follow-up question.")
        else:
            await self.publish(
                incident,
                f"*{incident.severity} {incident.id}* {incident.title}\n"
                f"Investigating `{incident.service}`.",
            )
            incident.record(f"Opened an incident thread in {incident.channel}.")
            agent: Agent = {
                "model": self.model,
                "instructions": INSTRUCTIONS,
                "reasoning": {"effort": "high"},
                "multi_agent": {"enabled": True, "max_concurrent_subagents": 3},
                "tools": configured_tools(),
            }
            session = await self.client.beta.agents.sessions.create(
                agent=agent,
                environment={
                    "type": "self_hosted",
                    "workspace_directory": "/workspace",
                },
            )
            incident.session_id = session.id
            if incident.status == "resolving":
                return ""
            environment = session.environment
            if environment.type != "self_hosted":
                raise RuntimeError("Expected a self-hosted execution environment.")
            incident.sandbox = await asyncio.to_thread(
                start_executor, environment.id, environment.remote_url
            )
            incident.record("Started an incident sandbox with AWS skills.")
            logger.info(
                "%s: sandbox %s, session %s",
                incident.id,
                incident.sandbox.short_id,
                session.id,
            )

        # The webhook handles propose_rollback; do not auto-complete its tool call.
        if incident.status == "resolving":
            return ""
        async with self.client.beta.agents.sessions.stream(
            session.id, input=prompt, tool_handlers=handlers
        ) as events:
            answer = await self.collect(events, incident)
        if incident.status == "resolving":
            return answer
        if question is not None and incident.analysis:
            incident.analysis += f"\n\nFollow-up: {question}\n{answer}"
        else:
            incident.analysis = answer
        if incident.status in {"investigating", "failed"}:
            incident.status = "investigated"
        await self.publish(incident, f"*Incident update*\n{answer}")
        incident.record(f"Posted investigation results to {incident.channel}.")
        return answer



Continue `agent.py`: `IncidentBot.collect`, `IncidentBot.request_approval`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/agent.py"
    async def collect(
        self, events: AsyncIterator[AgentSessionEvent], incident: Incident
    ) -> str:
        output: list[str] = []
        reported: set[str] = set()
        async for event in events:
            if incident.status == "resolving":
                return "".join(output)
            if event.type == "agent.session.environment.connected":
                incident.record("Connected the incident sandbox.")
            elif (
                event.type == "agent.session.turn.item.done" and event.item is not None
            ):
                if event.item.type == "command_execution":
                    logger.info(
                        "%s: sandbox command %s", incident.id, event.item.status
                    )
            elif (
                event.type == "agent.session.turn.item.added"
                and event.item.type == "function_call"
            ):
                tool_name = event.item.name
                if tool_name in PROGRESS and tool_name not in reported:
                    incident.record(PROGRESS[tool_name])
                    reported.add(tool_name)
            elif (
                event.type == "agent.session.subagent.created"
                and "specialist" not in reported
            ):
                incident.record("Delegated part of the investigation to a specialist.")
                reported.add("specialist")
            elif event.type == "agent.session.turn.output_text.delta":
                output.append(event.delta)
            elif event.type == "agent.session.turn.output_text.done" and not output:
                output.append(event.text)
            elif event.type in {
                "agent.session.failed",
                "agent.session.turn.failed",
                "error",
                "agent.session.environment.failed",
            }:
                raise RuntimeError(f"Incident investigation failed: {event.to_dict()}")
            elif event.type == "agent.session.turn.cancelled":
                if incident.status == "resolving":
                    return "".join(output)
                raise RuntimeError("Incident investigation was cancelled.")

        return "".join(output)

    async def request_approval(self, session_id: str) -> None:
        incident = next(
            (item for item in self.incidents.values() if item.session_id == session_id),
            None,
        )
        if incident is None:
            return
        # Independent of the stream's lock: the turn waits here for the Slack decision.
        async with incident.approval_lock:
            if incident.status in {"resolved", "resolving", "failed"}:
                return
            session = await self.client.beta.agents.sessions.retrieve(session_id)
            for action in session.required_actions:
                if action.type != "function_call" or action.name != "propose_rollback":
                    continue
                if (
                    incident.action is not None
                    and incident.action.get("call_id") == action.call_id
                    and incident.action.get("turn_id") == action.turn_id
                ):
                    if incident.slack_approval_ts is not None:
                        continue
                else:
                    try:
                        if incident.action is not None:
                            raise ValueError(
                                "A rollback has already been proposed for this incident."
                            )
                        arguments = action.arguments
                        if isinstance(arguments, str):
                            arguments = json.loads(arguments)
                        if not isinstance(arguments, dict):
                            raise ValueError("Rollback arguments must be an object.")
                        incident.action = self.propose_rollback(incident, arguments)
                    except (KeyError, ValueError):
                        await self.client.beta.agents.sessions.events.create(
                            session.id,
                            events=[
                                {
                                    "type": "agent.session.input.tool_result",
                                    "turn_id": action.turn_id,
                                    "call_id": action.call_id,
                                    "success": False,
                                    "error": "Invalid or duplicate rollback proposal.",
                                }
                            ],
                        )
                        continue
                    incident.action.update(
                        turn_id=action.turn_id, call_id=action.call_id
                    )
                approval = await self.publish(
                    incident,
                    f"Approval required: {incident.action['reason']}\n"
                    f"Roll back {incident.action['service']} to {incident.action['version']}?",
                    blocks=approval_blocks(incident),
                )
                incident.slack_approval_ts = str(approval["ts"])



Continue `agent.py`: `IncidentBot.decide_rollback`, `IncidentBot.submit_slack_decision`, `IncidentBot.incident`, `IncidentBot.resolve`, `IncidentBot._resolve`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/agent.py"
    async def decide_rollback(self, incident_id: str, approved: bool) -> dict[str, Any]:
        incident = self.incident(incident_id)
        async with incident.approval_lock:
            if incident.status in {"resolved", "resolving", "failed"}:
                raise ValueError("This incident is no longer awaiting a decision.")
            action = incident.action
            if action is None or incident.session_id is None:
                raise ValueError("No rollback is awaiting approval.")
            decision = "approved" if approved else "rejected"
            if action["status"] == decision:
                return action
            if action["status"] != "awaiting_approval":
                raise ValueError("This rollback request is no longer pending.")
            session = await self.client.beta.agents.sessions.retrieve(
                incident.session_id
            )
            if not any(
                pending.type == "function_call"
                and pending.call_id == action["call_id"]
                and pending.turn_id == action["turn_id"]
                for pending in session.required_actions
            ):
                raise ValueError("This rollback request is no longer pending.")
            await self.client.beta.agents.sessions.events.create(
                session.id,
                events=[
                    {
                        "type": "agent.session.input.tool_result",
                        "turn_id": str(action["turn_id"]),
                        "call_id": str(action["call_id"]),
                        "success": True,
                        "output": json.dumps(
                            {
                                "decision": decision,
                                "executed": False,
                                "service": action["service"],
                                "version": action["version"],
                            }
                        ),
                    }
                ],
            )
            action["status"] = decision
            incident.status = decision if approved else "investigated"
            incident.record(f"Rollback {decision}; returned the decision to the agent.")
            return action

    async def submit_slack_decision(self, incident_id: str, approved: bool) -> None:
        incident = self.incident(incident_id)
        try:
            decision = await self.decide_rollback(incident_id, approved)
        except Exception:
            logger.exception(
                "Could not submit the rollback decision for %s", incident_id
            )
            await self.publish(
                incident, "Could not submit the decision. Check the logs and retry."
            )
            return
        await self.publish(
            incident,
            f"Rollback {decision['status']}. The agent received the decision; no deployment ran.",
        )

    def incident(self, incident_id: str) -> Incident:
        incident = self.incidents.get(incident_id)
        if incident is None:
            raise ValueError(f"Unknown incident: {incident_id}")
        return incident

    async def resolve(self, incident_id: str) -> None:
        incident = self.incident(incident_id)
        async with incident.approval_lock:
            if incident.status == "resolved":
                return
            active = incident.lock.locked() or incident.status == "awaiting_approval"
            incident.status = "resolving"
            if active and incident.session_id is not None:
                await self.client.beta.agents.sessions.events.create(
                    incident.session_id, events=[{"type": "agent.session.input.cancel"}]
                )
        async with incident.lock:
            await self._resolve(incident)

    async def _resolve(self, incident: Incident) -> None:
        if incident.status == "resolved":
            return
        history: list[dict[str, Any]] = [
            *self.history,
            {
                "id": incident.id,
                "service": incident.service,
                "summary": incident.title,
                "root_cause": incident.analysis[:1200],
                "resolution": (
                    f"Rollback to {incident.action['version']} was approved; "
                    "execution was not recorded."
                    if incident.action is not None
                    and incident.action["status"] == "approved"
                    else "Resolved without a recorded rollback."
                ),
            },
        ]
        try:
            save_history(self.history_path, history)
            self.history = history
            incident.status = "resolved"
            if (
                incident.action is not None
                and incident.action["status"] == "awaiting_approval"
            ):
                incident.action["status"] = "expired"
            if incident.slack_thread_ts is not None:
                await self.publish(
                    incident, f"Resolved {incident.id}; saved the incident findings."
                )
        except OSError:
            if incident.status != "resolved":
                incident.status = "failed"
            raise
        finally:
            if incident.status == "resolved":
                if incident.slack_thread_ts is not None:
                    self.slack_threads.pop(incident.slack_thread_ts, None)
                self.fingerprints.pop(incident.fingerprint, None)
            await self.close_runtime(incident)
            incident.record("Closed the incident session and sandbox.")



Continue `agent.py`: `IncidentBot.close_runtime`, `IncidentBot.close`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/agent.py"
    async def close_runtime(self, incident: Incident) -> None:
        original_error = sys.exception()
        session_id = incident.session_id
        errors: list[Exception] = []
        if session_id is not None and session_id not in self.closed_sessions:
            try:
                await self.client.beta.agents.sessions.delete(session_id)
            except NotFoundError:
                self.closed_sessions.add(session_id)
                incident.session_id = None
            except Exception as error:
                errors.append(error)
            else:
                self.closed_sessions.add(session_id)
                incident.session_id = None
        if incident.sandbox is not None:
            try:
                await asyncio.to_thread(incident.sandbox.remove, force=True)
            except docker.errors.NotFound:
                incident.sandbox = None
            except Exception as error:
                errors.append(error)
            else:
                logger.info(
                    "%s: removed sandbox %s", incident.id, incident.sandbox.short_id
                )
                incident.sandbox = None
        if original_error is not None:
            for error in errors:
                original_error.add_note(f"Cleanup for session {session_id}: {error}")
        elif errors:
            raise ExceptionGroup(
                f"Could not clean up session {session_id} and its sandbox", errors
            )

    async def close(self) -> None:
        errors: list[Exception] = []
        for incident in self.incidents.values():
            try:
                await self.close_runtime(incident)
            except Exception as error:
                errors.append(error)
        if errors:
            raise ExceptionGroup("Could not close all incident runtimes", errors)


### main.py

Wire alert ingestion, signed OpenAI webhooks, and Slack callbacks into the FastAPI application.


In [ ]:
%%writefile "{application_dir}/main.py"
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "openai>=3.13.0",
#     "aiohttp",
#     "docker",
#     "fastapi",
#     "python-dotenv",
#     "slack-sdk",
#     "uvicorn",
# ]
# ///

"""Receive incident alerts, Agents API webhooks, and Slack callbacks."""

from __future__ import annotations

import hmac
import json
import logging
import os
import sys
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager
from pathlib import Path
from typing import Any
from urllib.parse import parse_qs

from dotenv import load_dotenv
from fastapi import BackgroundTasks, FastAPI, HTTPException, Request
from openai import AsyncOpenAI, InvalidWebhookSignatureError, OpenAI

# Support direct execution from any working directory.
if __package__ in {None, ""}:
    sys.path.insert(0, str(Path(__file__).resolve().parents[4]))


from examples.agents_api.apps.sev_bot.agent import IncidentBot
from examples.agents_api.apps.sev_bot.alerts import queue_alerts
from examples.agents_api.apps.sev_bot.slack import verify_slack_request

EXAMPLE_DIR = Path(__file__).resolve().parent


@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncIterator[None]:
    if not os.environ.get("OPENAI_WEBHOOK_SECRET"):
        raise RuntimeError(
            "Configure the Agents API webhook and set OPENAI_WEBHOOK_SECRET."
        )
    async with AsyncOpenAI() as client:
        app.state.bot = IncidentBot(client)
        try:
            yield
        finally:
            await app.state.bot.close()


app = FastAPI(title="SRE agent for incident response", lifespan=lifespan)


@app.post("/webhooks/alerts")
async def receive_alert(request: Request, tasks: BackgroundTasks) -> dict[str, Any]:
    expected = os.environ.get("ALERT_WEBHOOK_TOKEN")
    authorization = request.headers.get("authorization", "")
    if expected and not hmac.compare_digest(authorization, f"Bearer {expected}"):
        raise HTTPException(status_code=401, detail="Invalid alert webhook token.")

    payload = await request.json()
    if not isinstance(payload, dict):
        raise HTTPException(
            status_code=400, detail="Expected an incident webhook payload."
        )
    return queue_alerts(app.state.bot, payload, tasks)


@app.post("/webhooks/openai")
async def openai_webhook(request: Request) -> dict[str, Any]:
    secret = os.environ.get("OPENAI_WEBHOOK_SECRET")
    if not secret:
        raise HTTPException(status_code=500, detail="Set OPENAI_WEBHOOK_SECRET.")
    body = await request.body()
    try:
        with OpenAI(webhook_secret=secret) as verifier:
            verifier.webhooks.verify_signature(payload=body, headers=request.headers)
    except (InvalidWebhookSignatureError, ValueError) as error:
        raise HTTPException(
            status_code=401, detail="Invalid OpenAI webhook signature."
        ) from error
    try:
        event = json.loads(body)
    except ValueError as error:
        raise HTTPException(status_code=400, detail="Invalid webhook JSON.") from error
    if event.get("type") == "agent.session.action_required":
        data = event["data"]
        if data.get("required_action", {}).get("type") == "function_call":
            # Finish before acknowledging so a failed Slack delivery can be retried.
            await app.state.bot.request_approval(data["id"])
    return {"status": "ok"}




Continue `main.py`: `slack_events`, `slack_actions`, `main`, `Application entry point`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/main.py"
@app.post("/slack/events")
async def slack_events(request: Request, tasks: BackgroundTasks) -> dict[str, Any]:
    body = await request.body()
    verify_slack_request(body, request)
    payload = json.loads(body)
    if payload.get("type") == "url_verification":
        return {"challenge": str(payload["challenge"])}

    bot: IncidentBot = app.state.bot
    delivery = str(payload.get("event_id", ""))
    if delivery and delivery in bot.slack_deliveries:
        return {"status": "already_processed"}

    event = payload.get("event", {})
    if not isinstance(event, dict) or event.get("bot_id") or event.get("subtype"):
        return {"status": "ignored"}

    incident_id = bot.slack_threads.get(str(event.get("thread_ts", "")))
    if incident_id is None:
        return {"status": "ignored"}

    if delivery:
        bot.slack_deliveries.add(delivery)
    tasks.add_task(
        bot.investigate, bot.incident(incident_id), str(event.get("text", ""))
    )
    return {"status": "accepted"}


@app.post("/slack/actions")
async def slack_actions(request: Request, tasks: BackgroundTasks) -> dict[str, Any]:
    body = await request.body()
    verify_slack_request(body, request)
    encoded = parse_qs(body.decode()).get("payload", [])
    if not encoded:
        raise HTTPException(status_code=400, detail="Missing Slack action payload.")

    payload = json.loads(encoded[0])
    action = payload["actions"][0]
    bot: IncidentBot = app.state.bot
    incident_id = str(action["value"])
    try:
        if action["action_id"] not in {"approve_rollback", "reject_rollback"}:
            raise ValueError("Unknown Slack approval action.")
        bot.incident(incident_id)
    except ValueError as error:
        raise HTTPException(status_code=400, detail=str(error)) from error

    tasks.add_task(
        bot.submit_slack_decision,
        incident_id,
        action["action_id"] == "approve_rollback",
    )
    return {"text": "Submitting your decision to the agent."}


def main() -> None:
    import uvicorn

    load_dotenv(EXAMPLE_DIR / ".env")
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    logging.getLogger("httpx").setLevel(logging.WARNING)
    uvicorn.run(app, host="127.0.0.1", port=8003)


if __name__ == "__main__":
    main()


## Check the generated files

Compile all generated modules without importing them or contacting external services. This catches syntax errors before you launch the application.


In [ ]:
import py_compile

modules = ["alerts.py", "memory.py", "slack.py", "tools.py", "agent.py", "main.py"]
for filename in modules:
    py_compile.compile(str(application_dir / filename), doraise=True)
print(f"Compiled {len(modules)} application modules.")


## Launch the application (optional)

Edit the generated `.env` file with the credentials listed above. Install `uv` and, for sandbox applications, start Docker. The following cells are disabled by default. Enabling them may incur API usage and connect to the configured services.

Use the generated workspace for every path below. For a hosted deployment, package the generated application files and supply credentials through your deployment's secret configuration.


In [ ]:
import subprocess

BUILD_SANDBOX = False
if BUILD_SANDBOX:
    subprocess.run(
        ["docker", "build", "-t", "agent-api-sev-sandbox:latest", str(application_dir)],
        cwd=notebook_root,
        check=True,
    )


This application stays running to receive events. The process writes to `application.log` in the generated workspace. Inspect that file for startup failures and progress. Use the stop cell below when you finish.


In [ ]:
import os
import subprocess

RUN_APPLICATION = False
application_arguments = []
if RUN_APPLICATION:
    if application_process is not None and application_process.poll() is None:
        raise RuntimeError("Stop the previous application before launching again.")
    with (application_dir / "application.log").open("w") as application_log:
        application_process = subprocess.Popen(
            ["uv", "run", str(application_dir / "main.py"), *application_arguments],
            cwd=notebook_root,
            env={key: value for key, value in os.environ.items() if key != "VIRTUAL_ENV"},
            start_new_session=True,
            stdout=application_log,
            stderr=subprocess.STDOUT,
        )
    print(f"Process started: {application_process.pid}")
    print(f"Progress log: {application_dir / 'application.log'}")


After the receiver is ready and Slack credentials are configured, send the sample alert. This makes a real Slack post and starts an Agents API investigation using bundled telemetry. The approval flow records a decision without deploying a rollback.


In [ ]:
SEND_SAMPLE_ALERT = False
if SEND_SAMPLE_ALERT:
    subprocess.run(
        ["curl", "--fail-with-body", "-X", "POST", "http://127.0.0.1:8003/webhooks/alerts",
         "-H", "Content-Type: application/json", "--data-binary",
         f"@{application_dir / 'sample_alert.json'}"],
        check=True,
    )


### Stop a running application

Set `STOP_APPLICATION = True` after you finish. Interrupt the application process group so the application's shutdown handlers can close sessions and remove containers. Batch commands normally exit on their own. A timeout means shutdown is still in progress; inspect the log before taking further action.


In [ ]:
import os
import signal

STOP_APPLICATION = False
if STOP_APPLICATION and application_process is not None:
    if application_process.poll() is None:
        os.killpg(os.getpgid(application_process.pid), signal.SIGINT)
        application_process.wait(timeout=30)
    print(f"Application exited with status {application_process.returncode}.")


The generated workspace remains available for reports and memory. Remove it manually after stopping the application and saving any files you need. Do not rerun the launch cell while the previous process is running.


## Example result

The included checkout alert produces an investigation like this, using bundled telemetry and real Slack delivery.

The following illustrates a possible result; model-generated findings depend on the inputs and connected sources.

```text
SEV-1 Checkout API error rate reached 18.7%
Slack: #oncall
Evidence: bundled sample telemetry, not a live production diagnosis.

Customer impact: approximately 782 failed checkouts per minute.
GitHub: PR #418 / commit 8c41f2e reduced max_connections from 64 to 8.
AWS: CloudWatch alarm firing; ElastiCache has 143 waiting requests.
Memory: INC-0931 recorded the same Redis pool failure and recovery.
Action: Roll back checkout-api to 2026.08.26.3, pending approval.
```


## Next steps

- Connect the Slack app to `#oncall` and configure incident notifications from PagerDuty, incident.io, or Alertmanager.
- Replace fixture-backed evidence with your GitHub repository, AWS resources, production telemetry, and historical incident store.
- Persist incident fingerprints, Slack thread IDs, agent session IDs, and approved remediation decisions.
- Verify responder identity and production access before executing an approved rollback.


## Related documentation

- [Sandbox providers](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers): Replace local Docker with a third-party sandbox running the same image and executor.
- [GitHub MCP](https://github.com/github/github-mcp-server/blob/main/docs/remote-server.md): Configure GitHub's hosted MCP server, read-only mode, and tool selection.
- [AWS skills](https://github.com/aws/agent-toolkit-for-aws/tree/main/skills): Skill instructions for AWS observability, compute, databases, and incident investigation.
- [AWS Agent Toolkit](https://github.com/aws/agent-toolkit-for-aws): Official AWS agent skills, MCP servers, and operational integrations.
- [AWS DevOps Agent integration](https://github.com/aws/agent-toolkit-for-aws/tree/main/plugins/aws-agents-for-devsecops): AWS incident investigation, service inspection, and remediation recommendations through MCP.
- [Slack chat.postMessage](https://docs.slack.dev/reference/methods/chat.postMessage/): Send incident updates and follow-up messages to a Slack thread.


## Files

- [main.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/sev_bot/main.py): Webhook routes and application startup.
- [agent.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/sev_bot/agent.py): Incident investigation, approval, resolution, and cleanup.
- [alerts.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/sev_bot/alerts.py): Alert normalization and background task scheduling.
- [tools.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/sev_bot/tools.py): Service evidence and GitHub/AWS MCP tools.
- [slack.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/sev_bot/slack.py): Slack messages, buttons, and callback verification.
- [memory.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/sev_bot/memory.py): Incident history loading, search, and atomic saves.
